In [3]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization, InputLayer
import os
import random
import h5py

msvcp140.dll
msvcp140_1.dll


In [4]:

def get_dataset_name(file_name_with_dir):
    
    filename_without_dir = file_name_with_dir.split('/')[-1]
    print(filename_without_dir)
    temp = filename_without_dir.split('_')[:-1]
    print(temp)
    dataset_name = "_".join(temp)
    return dataset_name
filename_path="./data/Intra/train/rest_105923_1.h5"
with h5py. File (filename_path , 'r') as f :
    dataset_name = get_dataset_name(filename_path)
    print(dataset_name)
    matrix = f.get(dataset_name)[()]
    print(type(matrix ))
    print(matrix.shape)

rest_105923_1.h5
['rest', '105923']
rest_105923
<class 'numpy.ndarray'>
(248, 35624)


In [5]:
def z_score_normalization(data):
    mean = data.mean(axis=0)
    std = data.std(axis=0)
    return (data - mean) / std

In [6]:
def segment_data(data, label, window_size=500, stride=500):
    segments = []
    labels = []
    for start in range(0, data.shape[1] - window_size + 1, stride):
        end = start + window_size
        segment = data[:, start:end]
        segments.append(segment)
        labels.append(label)
    return segments, labels

In [7]:
def infer_label_from_filename(filename, label_map):
    filename = filename.lower().replace('\\', '/')
    basename = os.path.basename(filename)
    
    for key in label_map:
        if key in basename:
            return label_map[key]
    
    raise ValueError(f"Could not infer label from filename: {filename}")

In [8]:
def load_and_preprocess(filepath, label_map, window_size=500, stride=500, downsample_factor=20):
    filename = filepath.lower()
    task_label = infer_label_from_filename(filepath, label_map)
    # Load data
    with h5py.File(filepath, 'r') as f:
        datasetname = list(f.keys())[0]
        data = f.get(datasetname)[()]  # Shape: (248, 35624)

    
# Segment the data into overlapping windows
    segments = []
    labels = []
    num_timepoints = data.shape[1]

    for start in range(0, num_timepoints - window_size + 1, stride):
        end = start + window_size
        window = data[:, start:end]

        mean = window.mean(axis=1, keepdims=True)
        std = window.std(axis=1, keepdims=True)
        window = (window - mean) / (std + 1e-8)

        segments.append(window[..., np.newaxis])  # Add channel dim for CNN
        labels.append(task_label)

    
    return segments, labels

In [9]:
def data_generator(filepaths, label_map, batch_size=8):
    while True:  # Infinite generator
        all_segments, all_labels = [], []
        for filepath in filepaths:
            segments, labels = load_and_preprocess(filepath, label_map)
            all_segments.extend(segments)
            all_labels.extend(labels)

        X = np.array(all_segments)
        y = np.array(all_labels)

        indices = np.arange(len(y))
        np.random.shuffle(indices)
        X = X[indices]
        y = y[indices]

        for i in range(0, len(X), batch_size):
            yield X[i:i+batch_size], y[i:i+batch_size]

In [10]:
# Parameters
train_dir = './data/Cross/train'
batch_size = 8  # number of files per iteration
epochs_per_batch = 1  # how many epochs for each file batch
total_epochs = 10  # total desired training epochs
l2_lambda = 0.001  # regularization strength

# Load all file paths and shuffle
all_filepaths = [
    os.path.normpath(os.path.join(train_dir, fname))
    for fname in os.listdir(train_dir)
    if fname.endswith('.h5')
]
np.random.shuffle(all_filepaths)

# Label map
label_map = {
    'rest': 0,
    'math': 1,
    'story': 1,
    'story_math': 1,         
    'working_memory': 2,
    'memory': 2,
    'motor': 3
}


In [11]:
from tensorflow.keras.regularizers import l2

def build_cnn(input_shape=(248, 500, 1), num_classes=4, l2_lambda=0.001):
    model = Sequential([
        InputLayer(input_shape=input_shape),

        Conv2D(32, (3, 3), activation='relu', padding='same',
               kernel_regularizer=l2(l2_lambda)),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
       #  Dropout(0.3),
        Conv2D(64, (3, 3), activation='relu', padding='same',
               kernel_regularizer=l2(l2_lambda)),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
       #  Dropout(0.3),
        Conv2D(128, (3, 3), activation='relu', padding='same',
               kernel_regularizer=l2(l2_lambda)),
        BatchNormalization(),
        MaxPooling2D((2, 2)),

        Flatten(),
       #  Dense(128, activation='relu', kernel_regularizer=l2(l2_lambda)),
       #  Dropout(0.5),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

In [12]:
def load_test_data(test_filepaths, label_map, window_size=500, stride=500, downsample_factor=20):
    all_segments = []
    all_labels = []

    for filepath in test_filepaths:
        segments, labels = load_and_preprocess(
            filepath, label_map,
            window_size=window_size,
            stride=stride,
            downsample_factor=downsample_factor
        )
        all_segments.extend(segments)
        all_labels.extend(labels)

    X_test = np.array(all_segments)
    y_test = np.array(all_labels)
    return X_test, y_test


In [13]:
import os
import numpy as np
from sklearn.utils import shuffle

def chunked_training(model, train_filepaths, label_map, chunk_size=8, epochs=5, batch_size=8):
    num_chunks = len(train_filepaths) // chunk_size

    for epoch in range(epochs):
        print(f"\n=== Epoch {epoch+1}/{epochs} ===")
        shuffled_files = shuffle(train_filepaths)

        for i in range(num_chunks):
            chunk_files = shuffled_files[i * chunk_size : (i + 1) * chunk_size]
            print(f"\nTraining on files {i * chunk_size + 1} to {(i + 1) * chunk_size}")

            # Load and preprocess only this chunk
            X_batch, y_batch = load_test_data(chunk_files, label_map)

            # Fit model on this chunk
            model.fit(X_batch, y_batch, batch_size=batch_size, epochs=1, verbose=1)

    return model


In [15]:
from glob import glob

train_folder = './data/Cross/train'
train_filepaths = glob(os.path.join(train_folder, '*.h5'))
train_filepaths = [os.path.normpath(p) for p in train_filepaths]

model = build_cnn()  # or your smaller version
model = chunked_training(model, train_filepaths, label_map, chunk_size=8, epochs=10)



=== Epoch 1/10 ===

Training on files 1 to 8
71/71 [==============================] - 46s 636ms/step - loss: 0.8101 - accuracy: 0.7412

Training on files 9 to 16
71/71 [==============================] - 46s 653ms/step - loss: 1.1769 - accuracy: 0.7236

Training on files 17 to 24
71/71 [==============================] - 46s 646ms/step - loss: 0.7117 - accuracy: 0.7500

Training on files 25 to 32
71/71 [==============================] - 46s 652ms/step - loss: 0.8192 - accuracy: 0.7570

Training on files 33 to 40
71/71 [==============================] - 46s 650ms/step - loss: 0.4405 - accuracy: 0.8926

Training on files 41 to 48
71/71 [==============================] - 46s 652ms/step - loss: 0.2079 - accuracy: 0.9437

Training on files 49 to 56
71/71 [==============================] - 47s 662ms/step - loss: 0.1061 - accuracy: 0.9754

Training on files 57 to 64
71/71 [==============================] - 45s 639ms/step - loss: 0.3857 - accuracy: 0.9225

=== Epoch 2/10 ===

Training on files 

In [18]:
import glob

for i in range(1, 4):
    # Collect test files
    test_folder = f"./data/Cross/test{i}"
    test_filepaths = glob.glob(os.path.join(test_folder, '*.h5'))
    test_filepaths = [os.path.normpath(p) for p in test_filepaths]

    # Load and preprocess test data
    X_test, y_test = load_test_data(test_filepaths, label_map)
    # Option A: Direct evaluation
    loss, accuracy = model.evaluate(X_test, y_test, verbose=1)
    print(f"Test Accuracy: {accuracy}")

36/36 [==============================] - 18s 492ms/step - loss: 62.8383 - accuracy: 0.2958
Test Accuracy: 0.2957746386528015
36/36 [==============================] - 18s 503ms/step - loss: 59.6006 - accuracy: 0.2500
Test Accuracy: 0.25
36/36 [==============================] - 20s 564ms/step - loss: 62.9147 - accuracy: 0.2192
Test Accuracy: 0.21919013559818268
